# Bài 19 · Autoencoder & VAE: nén chữ số thành vài con số, rồi sinh chữ số mới từ đó

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/19-autoencoder-vae.ipynb)

Notebook đi kèm [Bài 19 của Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/autoencoder-vae/) (demo tương tác nằm ở trang bài học).
Ta tự viết bằng numpy PCA, một autoencoder và một VAE trên ảnh MNIST thật, rồi hỏi hai câu:
**nén ảnh 784 điểm xuống k số thì mất bao nhiêu**, và **lấy mã ở đâu để sinh một chữ số mới?**

**Bạn sẽ làm:**
1. Tính PCA từ đầu, đo sai số tái tạo theo k, và thấy một autoencoder tuyến tính học lại đúng không gian con đó.
2. Viết tay lan truyền ngược của autoencoder và VAE (có mẹo tái tham số hóa), kiểm tra bằng sai phân hữu hạn,
   rồi huấn luyện chúng với mã 2 chiều.
3. Giải mã bản đồ tiềm ẩn 2 chiều của demo, lấy mẫu từ $\mathcal N(0, I)$, từ một phân phối Gauss khớp với mã
   và từ một hỗn hợp Gauss, và xem mẫu rơi vào khoảng trống ở đâu.
4. Đo phương sai của mẹo tái tham số hóa so với ước lượng hàm điểm (REINFORCE) trên chính bộ giải mã đó.
5. Đọc kết quả quét β: ảnh mờ vì đâu, khi nào VAE sụp, và vì sao Stable Diffusion chỉ dùng autoencoder để nén.

**Cần biết trước:** [Bài 03 · Mạng nhiều lớp](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/mlp-va-xor/),
[Bài 04 · Softmax & hàm mất mát](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/ham-mat-mat/) (cross-entropy nhị phân)
và [Bài 06 · Lan truyền ngược](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/lan-truyen-nguoc/); bộ tối ưu Adam của
[Bài 07](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/bo-toi-uu/). **Thời gian chạy:** dưới 1 phút trên CPU, kể cả
tải MNIST (~11 MB) và 1,7 MB kết quả của demo lần đầu.

Mọi con số trong notebook đều do chính các ô bên dưới tính ra, và trùng với số trong demo. Riêng các lần huấn luyện
đầy đủ (20.000 bước × 3 seed cho mỗi cấu hình, khoảng 45 phút trên 16 nhân) được nạp từ đúng các tệp mà demo dùng;
notebook chạy lại một phiên bản nhỏ của từng cơ chế và kiểm tra những gì tính lại được.

> Các ô **🤔 Dự đoán trước** đặt một câu hỏi trước khi chạy. Hãy tự trả lời rồi mới chạy ô kế tiếp.

## 1 · Chuẩn bị

Chỉ cần numpy và matplotlib (Colab đã cài sẵn). Hàm `vn()` in số theo kiểu Việt Nam: dấu phẩy thập phân,
dấu chấm hàng nghìn. Python vẫn *gõ* số thập phân bằng dấu chấm (`0.1`), chỉ phần in ra là đổi.

In [ ]:
import base64
import gzip
import hashlib
import json
import math
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator, ScalarFormatter

# Bảng màu của khóa học (an toàn cho người mù màu); luôn đi kèm một mã hóa thứ hai (hình dạng điểm, nét, nhãn):
# từ mục 5, autoencoder = cam, ô vuông; VAE = xanh, chấm tròn
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
DIVERGING = LinearSegmentedColormap.from_list("cam_xam_xanh", [ORANGE, "#f2f2f2", BLUE])  # âm → 0 → dương
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def vn(x, digits=2):
    """Số kiểu Việt Nam: 13.007 · 0,1 · −4,38 (không in “−0”)."""
    s = f"{round(float(x), digits) + 0.0:,.{digits}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    return s.replace("-", "−")


def pct(p, digits=2):
    return vn(100 * p, digits) + "%"


def span(values, digits=2):
    """Khoảng min–max qua các seed, như demo: 27,31–27,71."""
    lo, hi = min(values), max(values)
    return vn(lo, digits) if round(lo, digits) == round(hi, digits) else f"{vn(lo, digits)}–{vn(hi, digits)}"


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def sci(x, digits=1):
    """Số rất nhỏ hoặc rất lớn: 1,9·10⁻⁷."""
    e = math.floor(math.log10(abs(x)))
    return f"{vn(x / 10**e, digits)}·10{str(e).translate(SUP)}"


def vn_tick(v, _pos=None):
    """Nhãn vạch trên trục: 0,5 · −2 · 60.000 (bỏ số 0 thừa ở cuối)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return vn(v, len(s.split(".")[1]) if "." in s else 0)


def log_tick(v, _pos=None):
    """Nhãn vạch trục log: 0,5 · 20 · 10⁻¹²."""
    if 0.01 <= v <= 1000:
        return vn_tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{vn_tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def vn_axes(fig):
    """Đổi nhãn mọi trục của hình (cả thanh màu) sang kiểu Việt Nam; gọi ngay trước plt.show().
    Trục log lấy vạch 1–2–5 nếu ngắn, hoặc các lũy thừa của 10 (cách đều, tối đa ~9 vạch) nếu dài."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                # only axes still on matplotlib's default number formatter: text labels and custom formatters stay
                if isinstance(axis.get_major_formatter(), ScalarFormatter):
                    axis.set_major_formatter(FuncFormatter(vn_tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            if hi / lo < 300:
                ticks = [m * 10.0**k for k in range(k0 - 1, k1 + 1) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
            else:
                step = max(1, math.ceil((k1 - k0) / 8))
                ticks = [10.0**k for k in range(k0, k1 + 1) if k % step == 0]
                if step > 1:
                    axis.set_minor_locator(NullLocator())
            axis.set_major_locator(FixedLocator(ticks))
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))


def show_digits(axes, images, titles=None, fontsize=9):
    """Vẽ mỗi ảnh 784 điểm (giá trị 0–1) vào một ô, nền trắng, nét đen."""
    for i, ax in enumerate(np.ravel(axes)):
        ax.axis("off")
        if i < len(images):
            ax.imshow(np.asarray(images[i]).reshape(28, 28), cmap="gray_r", vmin=0, vmax=1)
            if titles is not None:
                ax.set_title(titles[i], fontsize=fontsize)


def tile(images, rows, cols, gap=2):
    """Ghép rows × cols ảnh 28×28 thành một ảnh lớn (nhanh hơn nhiều so với một trục cho mỗi ảnh)."""
    out = np.zeros((rows * (28 + gap) - gap, cols * (28 + gap) - gap))
    for i, im in enumerate(images[: rows * cols]):
        r, c = divmod(i, cols)
        out[r * (28 + gap) : r * (28 + gap) + 28, c * (28 + gap) : c * (28 + gap) + 28] = np.reshape(im, (28, 28))
    return out


print(vn(13007, 0), vn(0.1, 1), vn(-4.3805, 4), pct(0.993527), span([27.307, 27.714]), sci(1.9e-7))

## 2 · Dữ liệu

MNIST (LeCun và cộng sự, 1998): 60.000 ảnh huấn luyện và 10.000 ảnh kiểm tra, chữ số viết tay xám 28×28. Ô dưới tải
4 tệp gốc và kiểm tra mã MD5, nên ai chạy cũng có đúng cùng một dữ liệu. Giống các lần chạy của demo, ta chia:
- **huấn luyện:** 55.000 ảnh đầu (PCA và các mạng học trên phần này);
- **kiểm định:** 5.000 ảnh cuối của tệp huấn luyện (demo chọn tốc độ học $\eta$ ở đây);
- **kiểm tra:** 10.000 ảnh của tệp kiểm tra, chỉ để báo kết quả.

Ô cũng nạp bốn tệp kết quả mà demo dùng (`fetch_json`, có kiểm tra MD5):
- `vae-compress.json`: PCA, autoencoder tuyến tính và autoencoder sâu với k = 1–64, 3 seed;
- `vae-latent.json`: các VAE và autoencoder 2 chiều, quét β, bộ giải mã Gauss, ước lượng gradient;
- `vae-2d-ae.json`, `vae-2d-vae.json`: hai bộ giải mã 2 chiều mà demo chạy trong trình duyệt (seed 0).

In [ ]:
URL = "https://ossci-datasets.s3.amazonaws.com/mnist/"
FILES = {  # mã MD5 chuẩn của MNIST
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
    "t10k-images-idx3-ubyte.gz": "9fb629c4189551a2d022fa330f9573f3",
    "t10k-labels-idx1-ubyte.gz": "ec29112dd5afa0611ce80d1b7f02629c",
}
DATA = Path("mnist")


def fetch():
    DATA.mkdir(exist_ok=True)
    for name, md5 in FILES.items():
        p = DATA / name
        if not p.exists():
            urllib.request.urlretrieve(URL + name, p)
        assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"


def load(kind):
    """Định dạng IDX: 16 byte đầu của tệp ảnh (8 byte của tệp nhãn) là phần mô tả, còn lại là các byte 0–255."""
    with gzip.open(DATA / f"{kind}-images-idx3-ubyte.gz") as f:
        images = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 784)
    with gzip.open(DATA / f"{kind}-labels-idx1-ubyte.gz") as f:
        labels = np.frombuffer(f.read(), np.uint8, offset=8)
    return images, labels


# Kết quả của các lần huấn luyện dài (cùng tệp với demo trên web). build.py thay COURSE_DATA_URL bằng thư mục data/
# của repo notebook công khai.
def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"
    return json.loads(p.read_text())


fetch()
train_img, train_lab = load("train")
test_img, test_lab = load("t10k")
comp = fetch_json("vae-compress.json", "964972ff4498d4d13d8d72710c2a8789")
lat = fetch_json("vae-latent.json", "67bbda5976a13be0e72011a223954aeb")
packed = {
    "ae": fetch_json("vae-2d-ae.json", "e292fd127fe3e6c7bff5937e3aad1d66"),
    "vae": fetch_json("vae-2d-vae.json", "ef1ececca80f225edaa8b94ce4c836b8"),
}
size = sum(
    (Path("data") / f).stat().st_size
    for f in ["vae-compress.json", "vae-latent.json", "vae-2d-ae.json", "vae-2d-vae.json"]
)
print("huấn luyện:", train_img.shape, " kiểm tra:", test_img.shape, f" · 4 tệp của demo: {vn(size / 1e6, 1)} MB")

Ảnh xám có điểm ảnh trong $[0, 1]$. Bộ giải mã Bernoulli của VAE (mục 5) mô hình hóa ảnh **nhị phân**: mỗi điểm ảnh
là 0 hoặc 1. Như các lần chạy của demo, ta nhị phân hóa ngẫu nhiên: điểm ảnh sáng $p$ thành 1 với xác suất $p$.
Ảnh huấn luyện được nhị phân hóa lại ở mỗi bước; ảnh kiểm tra thì nhị phân hóa **một lần** với seed 2, nên mọi con số
trên tập kiểm tra là trên cùng một tập ảnh 0/1. Ô dưới kiểm tra rằng 20 ảnh mà demo hiển thị có đúng từng bit như vậy.

In [ ]:
X_all = train_img.astype(np.float32) / 255
Xtr, Xval = X_all[:55000], X_all[55000:]  # huấn luyện · kiểm định
Xte, yte = test_img.astype(np.float32) / 255, test_lab.astype(np.int64)  # kiểm tra
Xte_bin = (np.random.default_rng(2).random((10000, 784), dtype=np.float32) < Xte).astype(np.float32)


def b64(s, dtype):
    """Giải mã một mảng mà demo lưu dạng base64 (little-endian)."""
    return np.frombuffer(base64.b64decode(s), dtype)


SHOW = lat["digits"]["idx"]  # 20 ảnh kiểm tra của demo: hai ảnh đầu tiên của mỗi chữ số
assert np.array_equal(b64(lat["digits"]["bits"], np.uint8).reshape(-1, 784), Xte_bin[SHOW])
print(f"huấn luyện {vn(len(Xtr), 0)} · kiểm định {vn(len(Xval), 0)} · kiểm tra {vn(len(Xte), 0)} ảnh")
print("chữ số của 20 ảnh demo:", [int(yte[i]) for i in SHOW])
print(f"{pct(Xte_bin.mean())} điểm ảnh nhị phân bằng 1; độ sáng trung bình của ảnh xám {pct(Xte.mean())}")

fig, axes = plt.subplots(2, 10, figsize=(10, 2.3))
show_digits(axes[0], Xte[SHOW[::2]], [f"xám: {yte[i]}" for i in SHOW[::2]])
show_digits(axes[1], Xte_bin[SHOW[::2]], ["nhị phân"] * 10)
plt.show()

Hàng trên là ảnh xám, hàng dưới là cùng ảnh đó sau khi nhị phân hóa: nét vẫn đọc được, nhưng mép nét lởm chởm vì
điểm ảnh xám nhạt đã thành 0 hoặc 1 một cách ngẫu nhiên.

## 3 · Nén tuyến tính: PCA từ đầu

Một ảnh là một vectơ hàng $x$ có 784 số. Nén nó còn $k$ số theo cách tuyến tính nghĩa là chọn $k$ hướng
$b_1, \dots, b_k$ (các hàng của ma trận $B_k$, cỡ $k \times 784$, trực chuẩn), lấy tọa độ của ảnh theo các hướng đó
rồi dựng lại:

$$z = (x - m)\,B_k^{\top} \quad (k \text{ số}), \qquad \hat x = m + z\,B_k, \qquad
\text{sai số} = \lVert \hat x - x \rVert^2 = \sum_{i=1}^{784} (\hat x_i - x_i)^2$$

với $m$ là ảnh trung bình. **PCA** chọn $k$ hướng cho tổng bình phương sai số nhỏ nhất trên tập huấn luyện: đó là $k$
vectơ riêng có trị riêng lớn nhất của ma trận hiệp phương sai $C$ (784 × 784), và trị riêng $\lambda_j$ là phương sai
của dữ liệu theo hướng $j$. Thước đo trong cả bài là sai số trên, tính trên ảnh xám và lấy trung bình trên 10.000 ảnh
kiểm tra, như Hinton & Salakhutdinov (2006).

Có hai cách tính cùng các hướng ấy: phân rã giá trị kỳ dị (SVD) của ma trận dữ liệu đã trừ trung bình, hoặc phân rã
trị riêng của $C$. Ô dưới kiểm tra rằng chúng trùng nhau trên 2.000 ảnh, rồi dùng cách thứ hai cho 55.000 ảnh: SVD của
ma trận 55.000 × 784 chậm hơn nhiều, còn $C$ chỉ là 784 × 784.

In [ ]:
KS = [1, 2, 4, 8, 16, 30, 64]


def scatter(X):
    """Số ảnh, tổng và ma trận XᵀX: đủ để tính trung bình và hiệp phương sai, và cộng dồn được giữa các tập."""
    X = X.astype(np.float64)
    return len(X), X.sum(0), X.T @ X


def pca_from(n, s, G):
    """Trung bình, các trị riêng (giảm dần) và các thành phần chính (hàng của B) từ scatter()."""
    m = s / n
    C = (G - n * np.outer(m, m)) / (n - 1)  # hiệp phương sai 784 × 784
    lam, V = np.linalg.eigh(C)  # eigh trả về trị riêng tăng dần
    return m, lam[::-1], V[:, ::-1].T


def pca_sse(m, B, X, k):
    """Sai số tái tạo (tổng bình phương trên 784 điểm ảnh) của từng ảnh, với k thành phần đầu."""
    Xc = X.astype(np.float64) - m
    R = (Xc @ B[:k].T) @ B[:k]
    return ((R - Xc) ** 2).sum(1)


# Hai cách, cùng một kết quả (trên 2.000 ảnh đầu)
S = Xtr[:2000].astype(np.float64)
_, sv, Vt = np.linalg.svd(S - S.mean(0), full_matrices=False)
_, lam_s, B_s = pca_from(*scatter(Xtr[:2000]))
cos10 = np.abs((Vt[:10] * B_s[:10]).sum(1))  # cos giữa hai hướng thứ j (dấu tùy ý)
assert np.allclose(sv[:10] ** 2 / 1999, lam_s[:10]) and np.allclose(cos10, 1)
print(f"SVD và trị riêng của C trên {vn(len(S), 0)} ảnh: 10 hướng đầu trùng nhau (|cos| nhỏ nhất {vn(cos10.min(), 6)})")

n55, s55, G55 = scatter(Xtr)  # dùng lại ở ô sau
m_pca, lam, B = pca_from(n55, s55, G55)
sse_pca = {k: pca_sse(m_pca, B, Xte, k).mean() for k in KS}
for k in KS:
    share = lam[:k].sum() / lam.sum()
    print(f"k = {k:>2}: sai số {vn(sse_pca[k], 2):>6} mỗi ảnh · giữ {pct(share, 1)} phương sai")
    assert abs(sse_pca[k] - comp["pca"]["sse"][k - 1]) < 2e-3, "khác với demo: hãy kiểm tra lại dữ liệu"
    assert abs(share - comp["pca"]["explained"][k - 1]) < 1e-3
print(f"không nén gì (chỉ ảnh trung bình m): sai số {vn(((Xte - m_pca) ** 2).sum(1).mean(), 2)}")

Với $k = 30$, PCA còn sai số 13,86 mỗi ảnh. Hinton & Salakhutdinov (2006) công bố 13,87 cho PCA 30 chiều; họ khớp
PCA trên cả 60.000 ảnh huấn luyện. Cộng thêm 5.000 ảnh kiểm định vào `scatter` là tái tạo được đúng con số ấy:

In [ ]:
n5, s5, G5 = scatter(Xval)
m60, _, B60 = pca_from(n55 + n5, s55 + s5, G55 + G5)
sse60 = pca_sse(m60, B60, Xte, 30).mean()
print(f"PCA khớp trên 60.000 ảnh, k = 30: {vn(sse60, 4)} → làm tròn {vn(sse60, 2)} (bài báo: 13,87)")
assert round(sse60, 2) == comp["hinton"]["pca"] == 13.87

Hình dưới so ảnh gốc (hàng trên) với ảnh PCA dựng lại từ $k$ = 2, 8 và 30 số. Hai số chỉ cho những vệt mờ, nhiều vệt
trông như một chữ số khác (số 7 thành một nét giống số 9); 8 số đã nhận ra được phần lớn chữ số; 30 số cho chữ số đọc
được, nhưng còn nhòe và có quầng xám quanh nét.

In [ ]:
fig, axes = plt.subplots(4, 10, figsize=(10, 4.4))
Xs = Xte[SHOW[::2]].astype(np.float64)
show_digits(axes[0], Xs)
for row, k in zip(axes[1:], [2, 8, 30]):
    show_digits(row, np.clip(m_pca + ((Xs - m_pca) @ B[:k].T) @ B[:k], 0, 1))
for row, label in zip(axes[:, 0], ["ảnh gốc", "PCA k = 2", "PCA k = 8", "PCA k = 30"]):
    row.text(-0.12, 0.5, label, transform=row.transAxes, ha="right", va="center", fontsize=9)
plt.subplots_adjust(left=0.1, wspace=0.05, hspace=0.08)
plt.show()

## 4 · Autoencoder tuyến tính học gì?

Một **autoencoder** gồm bộ mã hóa $x \mapsto z$ và bộ giải mã $z \mapsto \hat x$, học chung bằng gradient để sai số
tái tạo nhỏ nhất. Nút thắt $z$ chỉ có $k$ số, nên mạng phải chọn giữ lại gì. Trường hợp đơn giản nhất là cả hai
phía đều tuyến tính:

$$z = (x - m)\,W_e, \qquad \hat x = m + z\,W_d, \qquad W_e: 784 \times k,\;\; W_d: k \times 784$$

Baldi & Hornik (1989) chứng minh rằng điểm cực tiểu của sai số bình phương này là không gian con của PCA. Ta thử với
$k = 16$: khởi tạo ngẫu nhiên, Adam (Bài 07), batch 128, 1.500 bước, warmup rồi lịch cosin.

### 🤔 Dự đoán trước
Sau khi học, (a) sai số của autoencoder tuyến tính so với PCA $k = 16$ thế nào? (b) Mỗi hàng của $W_d$ có trùng với
một thành phần chính (|cos| ≈ 1) không?

In [ ]:
def adam_step(p, g, state, lr, b1=0.9, b2=0.999, eps=1e-8):
    """Adam (Bài 07), sửa p tại chỗ; state giữ m, v và số bước t."""
    state["t"] = t = state.get("t", 0) + 1
    for k in p:
        m = state.setdefault("m" + k, np.zeros_like(p[k]))
        v = state.setdefault("v" + k, np.zeros_like(p[k]))
        m += (1 - b1) * (g[k] - m)
        v += (1 - b2) * (g[k] ** 2 - v)
        p[k] -= (lr * (m / (1 - b1**t)) / (np.sqrt(v / (1 - b2**t)) + eps)).astype(p[k].dtype)


def lr_at(s, lr, steps):
    """Warmup tuyến tính 2% số bước đầu, rồi cosin về 0 (như các lần chạy của demo)."""
    warm = max(1, steps // 50)
    return lr * (s + 1) / warm if s < warm else lr * 0.5 * (1 + math.cos(math.pi * (s - warm) / (steps - warm)))


def train_linear_ae(k, steps=1500, lr=0.01, seed=0, batch=128):
    rng = np.random.default_rng(seed)
    m = Xtr.mean(0)
    p = {
        "We": (rng.standard_normal((784, k)) * np.sqrt(1 / 784)).astype(np.float32),
        "Wd": (rng.standard_normal((k, 784)) * np.sqrt(1 / k)).astype(np.float32),
    }
    state = {}
    for s in range(steps):
        xc = Xtr[rng.integers(0, len(Xtr), batch)] - m
        z = xc @ p["We"]
        r = z @ p["Wd"] - xc  # sai số tái tạo của từng điểm ảnh
        dr = 2 * r / batch  # ∂(trung bình của Σ r²)/∂r
        adam_step(p, {"Wd": z.T @ dr, "We": xc.T @ (dr @ p["Wd"].T)}, state, lr_at(s, lr, steps))
    return m, p


m_lin, p_lin = train_linear_ae(16)
Tc = Xte.astype(np.float64) - m_lin
sse_lin = ((Tc @ p_lin["We"] @ p_lin["Wd"] - Tc) ** 2).sum(1).mean()
print(f"k = 16, {vn(1500, 0)} bước: autoencoder tuyến tính {vn(sse_lin, 3)} · PCA {vn(sse_pca[16], 3)}")
# (b) so từng hàng của W_d với thành phần chính gần nó nhất
rows = p_lin["Wd"].astype(np.float64)
rows /= np.linalg.norm(rows, axis=1, keepdims=True)
best_cos = np.abs(rows @ B[:16].T).max(1)
print(f"|cos| giữa mỗi hàng của W_d và thành phần chính gần nhất: {vn(best_cos.min())}–{vn(best_cos.max())}")
# (c) so cả không gian con: cos của các góc chính giữa span(W_d) và span(16 thành phần chính đầu)
Q = np.linalg.qr(rows.T)[0]
cos_sub = np.linalg.svd(B[:16] @ Q, compute_uv=False)
print(f"góc chính giữa hai không gian con: cos nhỏ nhất {vn(cos_sub.min(), 4)}; trùng {pct((cos_sub**2).mean(), 1)}")
assert abs(sse_lin / sse_pca[16] - 1) < 0.005 and (cos_sub**2).mean() > 0.99 and best_cos.max() < 0.9

(a) Sai số gần như bằng PCA, và (c) không gian con mà $W_d$ trải ra trùng với không gian của 16 thành phần chính
đầu tiên tới 99,9%. Nhưng (b) không hàng nào của $W_d$ trùng với một thành phần chính: autoencoder tuyến tính học
đúng **không gian con** của PCA, không học đúng các **trục**. Quay các hướng bên trong không gian con đó (rồi quay
ngược ở bộ mã hóa) không đổi $\hat x$, nên sai số không có lý do gì để chọn một bộ trục riêng.

Vậy muốn nén tốt hơn PCA, phải có hàm kích hoạt phi tuyến. Hình dưới vẽ sai số theo $k$ (trục log) của PCA (tính ở
trên), autoencoder tuyến tính và autoencoder sâu 784-256-128-$k$ với bộ giải mã đối xứng, cả hai lấy từ các lần chạy
của demo (20.000 bước, 3 seed, mỗi seed một điểm), cùng hai số công bố của Hinton & Salakhutdinov ở $k = 30$
(hình thoi rỗng).

In [ ]:
lin = {int(k): [s["test"] for s in r["seeds"]] for k, r in comp["linear"].items()}
deep = {int(k): [s["test"] for s in r["seeds"]] for k, r in comp["deep"].items()}
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(KS, [sse_pca[k] for k in KS], color=BLUE, marker="o", lw=2, label="PCA (tính ở trên)")
for k in KS:
    ax.scatter([k] * 3, lin[k], s=70, facecolor="none", edgecolor=GREEN, lw=1.5, zorder=3)
    ax.scatter([k] * 3, deep[k], s=28, marker="s", color=ORANGE, zorder=3)
ax.scatter([], [], s=70, facecolor="none", edgecolor=GREEN, lw=1.5, label="autoencoder tuyến tính (3 seed)")
ax.scatter([], [], s=28, marker="s", color=ORANGE, label="autoencoder sâu 784-256-128-k (3 seed)")
ax.scatter([30, 30], [comp["hinton"]["pca"], comp["hinton"]["ae"]], s=80, marker="D", facecolor="none",
           edgecolor="black", label="Hinton & Salakhutdinov (2006): PCA, 784-1000-500-250-30")  # fmt: skip
ax.set_xscale("log")
ax.set(xlabel="k (số con số trong mã)", ylabel="sai số tái tạo mỗi ảnh kiểm tra", ylim=(0, 50))
ax.set_title("Nén còn k số: tuyến tính chỉ bằng PCA, phi tuyến tốt hơn nhiều")
ax.legend(frameon=False, fontsize=8)
vn_axes(fig)
plt.show()
for k in (2, 30):
    print(f"k = {k:>2}: PCA {vn(sse_pca[k])} · tuyến tính {span(lin[k])} · sâu {span(deep[k])}")
print(f"Hinton & Salakhutdinov, k = 30: PCA {vn(comp['hinton']['pca'])}, autoencoder sâu {vn(comp['hinton']['ae'])}")

Vòng tròn xanh lá (autoencoder tuyến tính) nằm đúng trên đường PCA ở mọi $k$. Ô vuông cam (autoencoder sâu) thấp hơn
hẳn: với 2 số, sai số giảm từ 43,64 xuống khoảng 27, với 30 số từ 13,86 xuống khoảng 3,6. Mạng của Hinton &
Salakhutdinov (784-1000-500-250-30) to hơn, được tiền huấn luyện từng lớp bằng RBM, và đạt 3,00.

## 5 · Autoencoder và VAE, viết tay

Từ đây, ta làm việc với ảnh nhị phân và một mạng nhỏ đủ chạy trong vài giây:

$$\text{mã hóa: } h = \mathrm{ReLU}(xW_1 + b_1),\;\; \mu = hW_\mu + b_\mu,\;\; \log\sigma^2 = hW_\sigma + b_\sigma
\qquad \text{giải mã: } g = \mathrm{ReLU}(zW_2 + b_2),\;\; a = gW_3 + b_3,\;\; \hat x = \mathrm{sigmoid}(a)$$

với lớp ẩn 128 nơ-ron mỗi phía (các mạng của demo là 784-256-128 và đối xứng). Bộ giải mã cho mỗi điểm ảnh một xác
suất $\hat x_i$, tức một phân phối Bernoulli $p(x \mid z)$, nên sai số tái tạo là cross-entropy nhị phân của Bài 04,
cộng trên 784 điểm ảnh:

$$-\log p(x \mid z) = -\sum_{i=1}^{784} \big[x_i \log \hat x_i + (1 - x_i)\log(1 - \hat x_i)\big]
\qquad \frac{\partial(-\log p)}{\partial a_i} = \hat x_i - x_i$$

- **Autoencoder:** $z = \mu$, mất mát là sai số tái tạo (không có $\log\sigma^2$).
- **VAE:** $z = \mu + \sigma \odot \varepsilon$ với $\varepsilon \sim \mathcal N(0, I)$ (mẹo tái tham số hóa), mất mát
  $\mathcal L = -\log p(x \mid z) + \beta\, D_{\mathrm{KL}}$ với
  $D_{\mathrm{KL}} = \tfrac12 \sum_j (\mu_j^2 + \sigma_j^2 - 1 - \log\sigma_j^2)$.

Vì $z$ là một hàm khả vi của $\mu$, $\sigma$ và một nhiễu $\varepsilon$ cố định, gradient đi qua bước lấy mẫu như
qua mọi lớp khác: $\partial z/\partial \mu = 1$ và $\partial z/\partial \log\sigma^2 = \varepsilon \odot \sigma / 2$.
KL cộng thêm $\partial D_{\mathrm{KL}}/\partial\mu = \mu$ và $\partial D_{\mathrm{KL}}/\partial \log\sigma^2 =
(\sigma^2 - 1)/2$. Ô dưới viết cả lượt tiến lẫn lượt lùi; `rec="gauss"` là bộ giải mã Gauss dùng ở mục 11.

In [ ]:
def init_net(d, H=128, vae=True, seed=0, dtype=np.float32):
    """He cho các lớp ReLU, 1/fan-in cho μ và lớp ra, 0,1 × mức đó cho log σ² (như demo); độ lệch b = 0."""
    rng = np.random.default_rng(seed)
    shapes = {"1": (784, H, 2 / 784), "mu": (H, d, 1 / H), "2": (d, H, 2 / d), "3": (H, 784, 1 / H)}
    if vae:
        shapes["lv"] = (H, d, 0.01 / H)
    p = {}
    for name, (a, b, var) in shapes.items():
        p["W" + name] = (rng.standard_normal((a, b)) * np.sqrt(var)).astype(dtype)
        p["b" + name] = np.zeros(b, dtype)
    return p


def sigmoid(a):
    return 0.5 * (1 + np.tanh(0.5 * a))  # không tràn số với |a| lớn


def bce(a, x):
    """−log p(x|z) của bộ giải mã Bernoulli với logit a, cộng trên các điểm ảnh (Bài 04)."""
    return (np.maximum(a, 0) - a * x + np.log1p(np.exp(-np.abs(a)))).sum(1)


def encode(p, x):
    h = np.maximum(x @ p["W1"] + p["b1"], 0)
    return h, h @ p["Wmu"] + p["bmu"], (h @ p["Wlv"] + p["blv"] if "Wlv" in p else None)


def decode_logits(p, z):
    g = np.maximum(z @ p["W2"] + p["b2"], 0)
    return g, g @ p["W3"] + p["b3"]


def loss_grad(p, x, eps, beta=1.0, rec="bern", s2=0.5):
    """Trung bình trên batch của sai số tái tạo và KL, cùng gradient của rec + β·KL theo mọi tham số."""
    B = len(x)
    h, mu, lv = encode(p, x)
    sd = None if lv is None else np.exp(0.5 * lv)
    z = mu if lv is None else mu + sd * eps  # mẹo tái tham số hóa
    g, a = decode_logits(p, z)
    if rec == "bern":
        r, da = bce(a, x), sigmoid(a) - x
    else:  # Gauss với phương sai cố định σ²: ‖x̂ − x‖² / (2σ²)
        y = sigmoid(a)
        r, da = ((y - x) ** 2).sum(1) / (2 * s2), (y - x) / s2 * y * (1 - y)
    kl = np.zeros(B) if lv is None else 0.5 * (mu**2 + sd**2 - 1 - lv).sum(1)
    G = {}
    da = da / B
    G["W3"], G["b3"] = g.T @ da, da.sum(0)
    dg = (da @ p["W3"].T) * (g > 0)
    G["W2"], G["b2"] = z.T @ dg, dg.sum(0)
    dz = dg @ p["W2"].T
    dmu = dz if lv is None else dz + beta * mu / B
    dh = dmu @ p["Wmu"].T
    if lv is not None:
        dlv = dz * eps * sd / 2 + beta * (sd**2 - 1) / 2 / B
        G["Wlv"], G["blv"] = h.T @ dlv, dlv.sum(0)
        dh += dlv @ p["Wlv"].T
    G["Wmu"], G["bmu"] = h.T @ dmu, dmu.sum(0)
    dh *= h > 0
    G["W1"], G["b1"] = x.T @ dh, dh.sum(0)
    return float(r.mean()), float(kl.mean()), G

Lượt lùi viết tay rất dễ sai một hệ số $\tfrac12$ hay một dấu. Như Bài 06, ta so mọi gradient với sai phân hữu hạn
trung tâm $\big(\mathcal L(\theta + h) - \mathcal L(\theta - h)\big) / 2h$ trên một mạng nhỏ bằng float64 (d = 3,
lớp ẩn 8 nơ-ron, 4 ảnh), với $\varepsilon$ giữ cố định: mẹo tái tham số hóa biến $\mathcal L$ thành một hàm tất định của
tham số. Ô dưới in sai số tương đối lớn nhất cho autoencoder, VAE Bernoulli và VAE Gauss.

In [ ]:
def gradcheck(vae, rec, beta=0.7, seed=0, h=1e-5):
    rng = np.random.default_rng(seed)
    p = init_net(3, H=8, vae=vae, seed=seed, dtype=np.float64)
    for k in p:
        p[k] += 0.1 * rng.standard_normal(p[k].shape)  # để độ lệch b khác 0 và có nơ-ron ReLU tắt
    x = rng.random((4, 784)) if rec == "gauss" else (rng.random((4, 784)) < 0.3).astype(float)
    eps = rng.standard_normal((4, 3))

    def loss():
        r, kl, _ = loss_grad(p, x, eps, beta, rec, s2=0.3)
        return r + beta * kl

    G = loss_grad(p, x, eps, beta, rec, s2=0.3)[2]
    worst = 0.0
    for k in p:
        flat = p[k].reshape(-1)  # một view: sửa flat là sửa p[k]
        for j in rng.choice(flat.size, min(6, flat.size), replace=False):
            old = flat[j]
            flat[j] = old + h
            up = loss()
            flat[j] = old - h
            down = loss()
            flat[j] = old
            num, ana = (up - down) / (2 * h), G[k].reshape(-1)[j]
            worst = max(worst, abs(num - ana) / max(1e-8, abs(num) + abs(ana)))
    return worst


for name, vae, rec in [("autoencoder", False, "bern"), ("VAE Bernoulli", True, "bern"), ("VAE Gauss", True, "gauss")]:
    err = gradcheck(vae, rec)
    print(f"{name:13s}: sai số tương đối lớn nhất {sci(err)}")
    assert err < 1e-5

Sai số tương đối cỡ $10^{-7}$ ở mọi tham số đã thử: lượt lùi đúng. (Không nhỏ hơn được nhiều, vì $\mathcal L$ là tổng
trên 784 điểm ảnh, cỡ vài trăm nat, nên phép trừ $\mathcal L(\theta + h) - \mathcal L(\theta - h)$ mất vài chữ số
có nghĩa; một lỗi thật trong lượt lùi cho sai số cỡ 1.) Giờ huấn luyện. Ô dưới định nghĩa `train()` (Adam, batch 64, warmup +
cosin, mỗi bước một batch ảnh nhị phân hóa mới và một $\varepsilon$ mới) và `evaluate()` trên 10.000 ảnh kiểm tra:
- **sai số tái tạo** $-\log p(x \mid z)$ (nat, với một $\varepsilon$ cố định) và **KL** (nat);
- **SSE**: giải mã $z = \mu$ rồi so với ảnh *xám*, đúng thước đo của mục 3 và 4.

## 6 · Nén còn 2 số: autoencoder và VAE

Ta huấn luyện hai mạng giống hệt nhau, nút thắt $d = 2$ để vẽ được mã lên mặt phẳng: một autoencoder và một VAE
(β = 1). Mỗi mạng 1.200 bước, vài giây; các mạng của demo chạy 20.000 bước.

### 🤔 Dự đoán trước
VAE phải trả thêm số hạng KL, và nó tái tạo từ một $z$ có nhiễu. Vậy khi giải mã $z = \mu$, ảnh của VAE có **mờ hơn**
(SSE lớn hơn rõ rệt) ảnh của autoencoder cùng kiến trúc không?

In [ ]:
def train(d, vae, steps=1200, lr=3e-3, batch=64, seed=0, beta=1.0, rec="bern", s2=0.5, H=128):
    rng = np.random.default_rng(1000 + seed)  # thứ tự ảnh, nhị phân hóa và ε
    p, state, hist = init_net(d, H, vae, seed), {}, []
    for s in range(steps):
        idx = rng.integers(0, len(Xtr), batch)
        x = (rng.random((batch, 784), dtype=np.float32) < Xtr[idx]).astype(np.float32) if rec == "bern" else Xtr[idx]
        eps = rng.standard_normal((batch, d), dtype=np.float32)
        r, kl, G = loss_grad(p, x, eps, beta, rec, s2)
        hist.append((r, kl))
        adam_step(p, G, state, lr_at(s, lr, steps))
    return p, np.array(hist)


def evaluate(p, rec="bern", s2=0.5, seed=3):
    X = Xte_bin if rec == "bern" else Xte
    _, mu, lv = encode(p, X)
    z = mu
    if lv is not None:
        z = mu + np.exp(0.5 * lv) * np.random.default_rng(seed).standard_normal(mu.shape).astype(np.float32)
    a = decode_logits(p, z)[1]
    r = bce(a, X).mean() if rec == "bern" else (((sigmoid(a) - X) ** 2).sum(1) / (2 * s2)).mean()
    unit_kl = np.zeros(mu.shape[1]) if lv is None else (0.5 * (mu**2 + np.exp(lv) - 1 - lv)).mean(0)
    sse = ((sigmoid(decode_logits(p, mu)[1]) - Xte) ** 2).sum(1).mean()
    active = int((mu.astype(np.float64).var(0) > 0.01).sum())  # chiều hoạt động (Burda và cộng sự, 2016)
    return {"rec": r, "kl": unit_kl.sum(), "unit_kl": unit_kl, "sse": sse, "mu": mu, "active": active}


nets, res = {}, {}
print(f"mỗi mạng {vn(1200, 0)} bước, batch 64, η = 0,003")
for name, vae in [("ae", False), ("vae", True)]:
    nets[name], _ = train(2, vae)
    res[name] = evaluate(nets[name])
    r = res[name]
    print(
        f"{name:>3}: SSE {vn(r['sse'])} · −log p(x|z) {vn(r['rec'], 1)} nat · KL {vn(r['kl'])} nat · "
        f"độ lệch chuẩn của mã ({vn(r['mu'][:, 0].std())}; {vn(r['mu'][:, 1].std())}) · "
        f"mã trải từ {vn(r['mu'].min(), 1)} đến {vn(r['mu'].max(), 1)}"
    )
gap = abs(res["ae"]["sse"] - res["vae"]["sse"]) / res["ae"]["sse"]
print(f"chênh lệch SSE: {pct(gap, 1)}")
assert gap < 0.05 and 0.6 < res["vae"]["mu"].std(0).min() and res["vae"]["mu"].std(0).max() < 1.8
assert np.abs(res["ae"]["mu"]).max() > 6

ae2, vae2 = lat["groups"]["ae2"]["seeds"], lat["groups"]["vae2"]["seeds"]
print(
    f"các mạng của demo (20.000 bước, 3 seed): SSE autoencoder {span([s['sse'] for s in ae2])}, "
    f"VAE {span([s['sse'] for s in vae2])}"
)

Không. Hai mạng dựng lại ảnh gần như tốt bằng nhau: lần chạy ngắn này lệch nhau vài phần trăm, và các mạng 20.000 bước của
demo cho SSE 27,34–27,65 (autoencoder) và 27,26–27,71 (VAE), trong dao động giữa các seed. Ở 2 chiều, **ảnh mờ là do
nút thắt**: chỉ 2 số thì không đủ để giữ nét riêng của từng ảnh, và bộ giải mã vẽ trung bình của những ảnh có mã gần
nhau. KL không phải thủ phạm chính ở đây.

Hai mạng khác nhau ở **chỗ đặt mã**. Hình dưới vẽ mã $\mu$ của 5.000 ảnh kiểm tra đầu tiên (chấm xám), mỗi chữ số
ghi ở trung vị mã của nó, cùng ba vòng tròn nét đứt bán kính 1, 2, 3 của $\mathcal N(0, I)$. Hai khung có thang đo
khác nhau.

In [ ]:
def latent_map(ax, mu, labels, title):
    ax.scatter(mu[:, 0], mu[:, 1], s=2, color=GREY, alpha=0.35, lw=0)
    for c in range(10):
        med = np.median(mu[labels == c], axis=0)
        ax.text(*med, str(c), ha="center", va="center", fontsize=12, weight="bold",
                bbox={"boxstyle": "circle,pad=0.15", "fc": "white", "ec": "black", "lw": 0.8})  # fmt: skip
    t = np.linspace(0, 2 * np.pi, 200)
    for r in (1, 2, 3):
        ax.plot(r * np.cos(t), r * np.sin(t), color="black", ls="--", lw=0.8)
    ax.set(xlabel="z₁", ylabel="z₂", title=title)
    ax.set_aspect("equal")


fig, axes = plt.subplots(1, 2, figsize=(10, 4.6))
for ax, name in zip(axes, ["ae", "vae"]):
    latent_map(
        ax, res[name]["mu"][:5000], yte[:5000], {"ae": "autoencoder", "vae": "VAE"}[name] + " (lần chạy ngắn ở trên)"
    )
plt.tight_layout()
vn_axes(fig)
plt.show()
for name in ["ae", "vae"]:
    inside = (np.linalg.norm(res[name]["mu"], axis=1) < 2).mean()
    print(f"{name:>3}: {pct(inside, 1)} mã nằm trong vòng bán kính 2 (với N(0, I) là {pct(1 - math.exp(-2), 1)})")

Mã của autoencoder có thang đo tùy ý và trải thành những dải dài: không có gì trong mất mát của nó nói mã phải nằm ở
đâu. Mã của VAE gom quanh gốc với độ lệch chuẩn gần 1, vì KL phạt mọi $\mu_j$ xa 0 và mọi $\sigma_j$ khác 1. Đó là
điều ta cần để **sinh** ảnh: lấy $z$ từ $\mathcal N(0, I)$ rồi giải mã.

## 7 · Bản đồ tiềm ẩn của demo: giải mã, nội suy, lấy mẫu

Từ đây ta dùng hai bộ giải mã 2 chiều mà demo chạy trong trình duyệt: cùng kiến trúc 2 → 128 → 256 → 784 của các lần
chạy đầy đủ (20.000 bước, seed 0). Để tệp nhỏ, các lớp ẩn được lưu bằng float16 và ma trận lớp ra bằng int8 với một
hệ số tỉ lệ float16 cho mỗi cột; ô dưới giải nén chúng. Tệp cũng có $\mu$ và $\sigma$ của 20 ảnh demo, và
$\log p(x \mid z = \mu)$ mà Python đã tính cho từng ảnh: ta tính lại con số đó để kiểm tra bộ giải mã của mình.

In [ ]:
def unpack(t):
    if "f16" in t:
        return b64(t["f16"], "<f2").astype(np.float64).reshape(t["shape"])
    scale = b64(t["scale"], "<f2").astype(np.float64)  # một hệ số cho mỗi cột
    return b64(t["i8"], np.int8).astype(np.float64).reshape(t["shape"]) * scale


dec = {name: {k: unpack(t) for k, t in packed[name]["dec"].items()} for name in packed}


def decode(m, z):
    """E[x|z] = sigmoid(logit) của bộ giải mã đã giải nén: 784 xác suất cho mỗi z."""
    h1 = np.maximum(np.atleast_2d(z) @ m["d0.W"] + m["d0.b"], 0)
    h2 = np.maximum(h1 @ m["d1.W"] + m["d1.b"], 0)
    return sigmoid(h2 @ m["out.W"] + m["out.b"])


def loglik_grad(m, z, x):
    """log p(x|z) của bộ giải mã Bernoulli cho một ảnh nhị phân x và nhiều z, cùng gradient theo z (lan truyền ngược)."""
    h1 = np.maximum(z @ m["d0.W"] + m["d0.b"], 0)
    h2 = np.maximum(h1 @ m["d1.W"] + m["d1.b"], 0)
    a = h2 @ m["out.W"] + m["out.b"]
    dh2 = ((x - sigmoid(a)) @ m["out.W"].T) * (h2 > 0)  # ∂ log p / ∂a = x − x̂
    dh1 = (dh2 @ m["d1.W"].T) * (h1 > 0)
    return -bce(a, x[None]), dh1 @ m["d0.W"].T


print({k: v.shape for k, v in dec["vae"].items()})
D20 = lat["digits"]
mu20, sd20 = np.array(D20["vae"]["mu"]), np.array(D20["vae"]["sd"])
f_mu = np.array([loglik_grad(dec["vae"], mu20[i][None], Xte_bin[SHOW[i]].astype(np.float64))[0][0] for i in range(20)])
want = np.array([g["fMu"] for g in D20["vae"]["grad"]])
print(
    f"log p(x | z = μ) của 20 ảnh demo: {vn(f_mu.min(), 1)} … {vn(f_mu.max(), 1)} nat; lệch Python tối đa {sci(np.abs(f_mu - want).max())}"
)
assert np.abs(f_mu - want).max() < 1e-3, "khác với demo: hãy kiểm tra lại bộ giải mã"

Bộ giải mã đã giải nén cho đúng $\log p(x \mid z)$ mà Python tính khi xuất tệp (lệch dưới $10^{-3}$ nat). Tệp còn có
mã $\mu$ của 5.000 ảnh kiểm tra đầu tiên với cả hai mô hình. Hình dưới giải mã một lưới 13 × 13 điểm $z$, trải từ
phân vị 1% đến phân vị 99% của các mã trên mỗi trục: mỗi ô là ảnh $\mathbb E[x \mid z]$ tại tọa độ ghi trên trục.

In [ ]:
codes = {name: b64(lat["map"]["codes"][name], "<f2").astype(np.float64).reshape(-1, 2) for name in ("ae", "vae")}
code_lab = b64(lat["map"]["labels"], np.uint8)
assert np.array_equal(code_lab, yte[:5000])
NAME = {"ae": "autoencoder", "vae": "VAE"}
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, name in zip(axes, ["ae", "vae"]):
    lo, hi = np.quantile(codes[name], [0.01, 0.99], axis=0)
    g1, g2 = np.linspace(lo[0], hi[0], 13), np.linspace(hi[1], lo[1], 13)  # hàng trên cùng là z₂ lớn nhất
    tiles = decode(dec[name], np.array([[a, b] for b in g2 for a in g1])).reshape(13, 13, 28, 28)
    ax.imshow(tiles.transpose(0, 2, 1, 3).reshape(13 * 28, 13 * 28), cmap="gray_r", vmin=0, vmax=1,
              extent=[lo[0], hi[0], lo[1], hi[1]], aspect="auto")  # fmt: skip
    ax.set(xlabel="z₁", ylabel="z₂", title=f"{NAME[name]}: giải mã một lưới z")
    sse = ((decode(dec[name], codes[name]) - Xte[:5000]) ** 2).sum(1).mean()
    print(f"{NAME[name]:>11}: phân vị 1%–99% của mã: z₁ ∈ [{vn(lo[0], 1)}; {vn(hi[0], 1)}], z₂ ∈ [{vn(lo[1], 1)}; {vn(hi[1], 1)}]; "
          f"SSE của 5.000 ảnh dựng lại {vn(sse)}")  # fmt: skip
plt.tight_layout()
vn_axes(fig)
plt.show()

Cả hai bản đồ đều liên tục: đi một bước nhỏ trên mặt phẳng, chữ số đổi dần sang chữ số bên cạnh, và mọi chữ số đều có
chỗ của nó. Khác biệt là thang đo: mã của autoencoder trải trên một vùng rộng gấp mấy lần vùng $[-3; 3]^2$ mà
$\mathcal N(0, I)$ thường rơi vào, còn mã của VAE nằm gọn trong đó.

Một điều dễ quên: các ô trên là **trung bình** $\mathbb E[x \mid z]$, tức xác suất từng điểm ảnh bằng 1, không phải một
mẫu của mô hình. Một mẫu thật của $p(x \mid z)$ lấy mỗi điểm ảnh là 0 hay 1 theo xác suất đó:

In [ ]:
z0 = mu20[0]  # mã của ảnh demo đầu tiên (một số 7) trong VAE
p0 = decode(dec["vae"], z0)[0]
rng = np.random.default_rng(5)
fig, axes = plt.subplots(1, 5, figsize=(7, 1.8))
show_digits(axes, [Xte_bin[SHOW[0]], p0] + [(rng.random(784) < p0).astype(float) for _ in range(3)],
            ["ảnh nhị phân x", "E[x|z]", "mẫu 1", "mẫu 2", "mẫu 3"])  # fmt: skip
plt.show()
print(f"z = μ = ({vn(z0[0])}; {vn(z0[1])}), σ = ({vn(sd20[0][0], 3)}; {vn(sd20[0][1], 3)})")

Mẫu Bernoulli lấm tấm: mỗi điểm ảnh được tung đồng xu riêng, độc lập với các điểm bên cạnh. Bộ giải mã này không mô
hình hóa được mối liên hệ giữa các điểm ảnh, nên "ảnh mịn" mà hầu hết các bài về VAE trình bày là trung bình, và một
phần độ mờ của nó đến từ chính việc lấy trung bình.

### Nội suy
Đi thẳng từ mã của một ảnh đến mã của ảnh khác rồi giải mã từng điểm trên đường đi (hàng trên: autoencoder, hàng
dưới: VAE; từ số 7 đến số 0 trong 20 ảnh demo):

In [ ]:
mus = {"ae": np.array(D20["ae"]["mu"]), "vae": mu20}
a_i, b_i = 0, 6  # ảnh demo thứ 0 (số 7) và thứ 6 (số 0)
fig, axes = plt.subplots(2, 9, figsize=(10, 2.6))
for row, name in zip(axes, ["ae", "vae"]):
    path = [(1 - t) * mus[name][a_i] + t * mus[name][b_i] for t in np.linspace(0, 1, 9)]
    show_digits(row, decode(dec[name], np.array(path)))
    row[0].text(-0.15, 0.5, NAME[name], transform=row[0].transAxes, ha="right", va="center", fontsize=9)
plt.subplots_adjust(left=0.1, wspace=0.05)
plt.show()
mid = {n: [s["mid"] for s in lat["groups"][n + "2"]["seeds"]] for n in ("ae", "vae")}
rec_fd = [s["recFd"] for n in ("ae", "vae") for s in lat["groups"][n + "2"]["seeds"]]
print(f"FD của điểm giữa 10.000 cặp ảnh (3 seed): autoencoder {span(mid['ae'])}, VAE {span(mid['vae'])}; "
      f"ảnh dựng lại {span(rec_fd)}; hai tập ảnh thật {vn(lat['judge']['floor'])}")  # fmt: skip

Cả hai đều nội suy được: đường đi đi qua những chữ số nằm giữa 7 và 0 trên bản đồ. Để đo bằng số, demo dùng **FD**
(khoảng cách Fréchet): cách tính của FID (Heusel và cộng sự, 2017), nhưng thay mạng Inception bằng mạng LeNet của
Bài 12, nên các giá trị FD chỉ so được với nhau trong khóa học. FD càng nhỏ, tập ảnh càng giống ảnh thật; hai tập ảnh
thật cho FD 1,32. Điểm giữa của cả hai mô hình đều kém giống ảnh thật hơn ảnh dựng lại. VAE tốt hơn một chút (hai bộ
3 seed không chồng lên nhau), nhưng khoảng cách nhỏ hơn nhiều so với khoảng cách giữa điểm giữa và ảnh dựng lại.

## 8 · Lấy mẫu: $\mathcal N(0, I)$, phân phối Gauss khớp với mã, hay hỗn hợp Gauss?

Muốn sinh một chữ số mới, ta lấy một điểm $z$ rồi giải mã. Câu hỏi là lấy $z$ từ phân phối nào. Ta thử ba cách trên
cả hai bản đồ:
- **$\mathcal N(0, I)$**, phân phối tiên nghiệm của VAE;
- **phân phối Gauss khớp với mã**: cùng trung bình và hiệp phương sai với các mã;
- **hỗn hợp 10 Gauss (GMM)** khớp với các mã bằng thuật toán EM, *sau khi* đã huấn luyện xong (Ghosh và cộng sự gọi
  đó là ước lượng mật độ hậu kỳ, *ex-post density estimation*).

Ô dưới viết EM cho GMM với hiệp phương sai đầy đủ, khởi tạo bằng k-means++. Mỗi vòng có hai bước:
- **bước E:** tính xác suất mỗi thành phần đã sinh ra mỗi điểm;
- **bước M:** ước lượng lại trọng số, trung bình và hiệp phương sai của từng thành phần từ các xác suất đó.

Ta khớp trên mã của 4.000 ảnh kiểm tra đầu tiên và giữ 1.000 ảnh còn lại để định nghĩa **khoảng trống**: một mẫu
rơi vào khoảng trống nếu nó cách mã gần nhất xa hơn 99% số mã giữ lại. Ta cũng ghi nhãn của mã gần nhất, để xem mẫu
có phủ đủ 10 chữ số không.

In [ ]:
def gmm_fit(Z, K=10, iters=100, seed=0):
    rng = np.random.default_rng(seed)
    N, d = Z.shape
    centers = [Z[rng.integers(N)]]
    for _ in range(K - 1):  # k-means++: chọn tâm mới xa các tâm đã có
        d2 = np.min(((Z[:, None] - np.array(centers)[None]) ** 2).sum(-1), 1)
        centers.append(Z[rng.choice(N, p=d2 / d2.sum())])
    mu, cov, w = np.array(centers), np.array([np.cov(Z, rowvar=False)] * K), np.full(K, 1 / K)
    for _ in range(iters):
        logp = np.empty((N, K))  # bước E: log [w_k · N(z | μ_k, Σ_k)], bỏ hằng số chung
        for k in range(K):
            L = np.linalg.cholesky(cov[k])
            u = np.linalg.solve(L, (Z - mu[k]).T)
            logp[:, k] = np.log(w[k]) - 0.5 * (u**2).sum(0) - np.log(np.diag(L)).sum()
        R = np.exp(logp - logp.max(1, keepdims=True))
        R /= R.sum(1, keepdims=True)  # trách nhiệm: thành phần k đã sinh ra điểm n với xác suất R[n, k]
        Nk = R.sum(0) + 1e-10  # bước M
        w, mu = Nk / N, (R.T @ Z) / Nk[:, None]
        for k in range(K):
            Dk = Z - mu[k]
            cov[k] = (R[:, k, None] * Dk).T @ Dk / Nk[k] + 1e-6 * np.eye(d)
    return w, mu, cov


def gmm_sample(g, n, rng):
    w, mu, cov = g
    comp = rng.choice(len(w), n, p=w)
    return np.concatenate([rng.multivariate_normal(mu[k], cov[k], (comp == k).sum()) for k in range(len(w))])


def nearest(Q, R):
    """Khoảng cách tới điểm gần nhất trong R, và chỉ số của nó, cho mỗi hàng của Q."""
    out = [((Q[i : i + 500, None] - R[None]) ** 2).sum(-1) for i in range(0, len(Q), 500)]
    return np.sqrt(np.concatenate([o.min(1) for o in out])), np.concatenate([o.argmin(1) for o in out])


SAMPLERS = {"prior": "N(0, I)", "gauss": "Gauss khớp", "gmm": "GMM 10 thành phần"}
samples, live = {}, {}
for name in ("ae", "vae"):
    ref, held = codes[name][:4000], codes[name][4000:]
    r99 = np.quantile(nearest(held, ref)[0], 0.99)
    print(f"{NAME[name]}: khớp trên {vn(len(ref), 0)} mã, {vn(2000, 0)} mẫu mỗi cách; khoảng trống = xa mã gần nhất "
          f"hơn {vn(r99, 3)} (phân vị 99% của {vn(len(held), 0)} mã giữ lại)")  # fmt: skip
    rng = np.random.default_rng(7)
    samples[name] = {
        "prior": rng.standard_normal((2000, 2)),
        "gauss": rng.multivariate_normal(ref.mean(0), np.cov(ref, rowvar=False), 2000),
        "gmm": gmm_sample(gmm_fit(ref), 2000, rng),
    }
    for k, z in samples[name].items():
        dist, j = nearest(z, ref)
        share = np.bincount(code_lab[j], minlength=10) / len(z)
        live[name, k] = {"hole": (dist > r99).mean(), "rare": share.min(), "rare_cls": int(share.argmin())}
        print(
            f"{NAME[name]:>11}, z ~ {SAMPLERS[k]:<17}: {pct(live[name, k]['hole'], 1):>5} mẫu rơi vào khoảng trống; "
            f"chữ số hiếm nhất ({live[name, k]['rare_cls']}) chiếm {pct(live[name, k]['rare'], 1)}"
        )
assert live["ae", "prior"]["rare"] < 0.01 and live["ae", "gauss"]["hole"] > max(0.03, 3 * live["ae", "prior"]["hole"])
assert live["vae", "prior"]["hole"] < 0.05 and live["vae", "prior"]["rare"] > 0.03

Lấy $z$ từ $\mathcal N(0, I)$ cho autoencoder thì gần như không mẫu nào rơi vào khoảng trống, nhưng gần như không mẫu
nào là số 1: vòng tròn bán kính 3 quanh gốc chỉ phủ một góc nhỏ của bản đồ autoencoder. Với VAE, $\mathcal N(0, I)$
phủ đủ 10 chữ số. Hình dưới vẽ 24 mẫu của mỗi cách (hàng trên: autoencoder; hàng dưới: VAE).

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(10, 5.4))
for r, name in enumerate(("ae", "vae")):
    for c, k in enumerate(SAMPLERS):
        axes[r, c].imshow(tile(decode(dec[name], samples[name][k][:24]), 4, 6), cmap="gray_r", vmin=0, vmax=1)
        axes[r, c].set_title(f"{NAME[name]}, z ~ {SAMPLERS[k]}", fontsize=10)
        axes[r, c].axis("off")
plt.tight_layout()
plt.show()

### Chỉ cần đúng thang đo?
Phân phối Gauss khớp có đúng trung bình và hiệp phương sai của các mã autoencoder. Vậy mẫu của nó có giống chữ số thật
hơn hẳn mẫu từ $\mathcal N(0, I)$ không? Ô dưới in FD của 10.000 mẫu cho mỗi cách lấy mẫu, từ các lần chạy đầy đủ
của demo (3 seed), cùng FD của ảnh dựng lại.

In [ ]:
G = lat["groups"]


def fd(name, k):
    return [s[k]["fd"] for s in G[name]["seeds"]]


def holes(name, k):
    return [s[k]["hole"] for s in G[name]["seeds"]]


def compare(na, a, nb, b):
    """Như demo: hai bộ 3 seed chồng lên nhau thì chưa nói được bên nào tốt hơn (FD nhỏ hơn là tốt hơn)."""
    if min(b) <= max(a) and min(a) <= max(b):
        return "trong dao động giữa các seed"
    return f"{na if max(a) < min(b) else nb} tốt hơn"


print("FD của 10.000 mẫu so với ảnh kiểm tra (3 seed, các lần chạy đầy đủ của demo; hai tập ảnh thật cho 1,32):")
for n in ("ae2", "vae2"):
    row = " · ".join(f"{SAMPLERS[k]} {span(fd(n, k))}" for k in SAMPLERS)
    print(f"  {NAME[n[:-1]]:>11}: dựng lại {span([s['recFd'] for s in G[n]['seeds']])} · {row}")
gap_ae = [100 * h for h in holes("ae2", "gauss")], [100 * h for h in holes("ae2", "prior")]
print(f"autoencoder: {compare('Gauss khớp', fd('ae2', 'gauss'), 'N(0, I)', fd('ae2', 'prior'))}; mẫu rơi vào "
      f"khoảng trống {span(gap_ae[0], 1)}% với Gauss khớp, {span(gap_ae[1], 1)}% với N(0, I)")  # fmt: skip
print(
    f"autoencoder + GMM so với VAE + N(0, I): {compare('autoencoder + GMM', fd('ae2', 'gmm'), 'VAE', fd('vae2', 'prior'))}"
)
print(f"VAE: {compare('GMM', fd('vae2', 'gmm'), 'N(0, I)', fd('vae2', 'prior'))}")
GHOSH = {"AE, N(0, I)": 58.73, "AE + GMM": 10.66, "VAE, N(0, I)": 19.21}  # Ghosh và cộng sự (2020), Bảng 1, MNIST, FID
print("Ghosh và cộng sự (2020), FID trên MNIST: " + " · ".join(f"{k} {vn(v)}" for k, v in GHOSH.items()))

Không. Phân phối Gauss khớp có đúng thang đo của các mã, nhưng các mã không có hình một phân phối Gauss: chúng nằm
thành cụm và dải, xen những vùng trống. Phân phối Gauss phủ cả những vùng đó, nơi bộ giải mã chưa từng được huấn luyện
(4,5–7,1% mẫu rơi vào khoảng trống), nên FD không tốt hơn $\mathcal N(0, I)$ rõ rệt: hai bộ 3 seed chồng lên nhau.

Hỗn hợp 10 Gauss thì bám theo các cụm, và autoencoder + GMM cho FD ngang VAE lấy mẫu từ $\mathcal N(0, I)$: trong dao
động giữa các seed. Ghosh và cộng sự (2020) thấy điều tương tự trên MNIST với FID thật: autoencoder lấy từ
$\mathcal N(0, I)$ cho 58,73, thêm GMM còn 10,66, so với 19,21 của VAE lấy từ $\mathcal N(0, I)$ (Bảng 1 của họ).
Ưu điểm thật của VAE là nó **có sẵn** một phân phối để lấy mẫu; autoencoder phải học thêm bước đó.

Ngay cả VAE cũng lấy mẫu tốt hơn khi khớp GMM trên mã của chính nó. Trung bình trên các ảnh, KL tách thành hai phần
(Hoffman & Johnson, 2016): thông tin mà mã mang về ảnh, cộng với khoảng cách từ **phân phối chung của các mã**
$q(z) = \mathbb E_x\, q(z \mid x)$ đến $\mathcal N(0, I)$. Phần thứ hai không bao giờ về 0 hẳn, nên $\mathcal N(0, I)$
vẫn rơi vào những vùng mà ít mã nào ở đó (Dai & Wipf, 2019). Với mã nhiều chiều hơn, khoảng cách ấy lớn hơn:

In [ ]:
for n, label in [("vae2", "d = 2"), ("d16", "d = 16"), ("b1", "d = 50")]:
    print(f"VAE {label:>6}: FD với N(0, I) {span(fd(n, 'prior'))} · với GMM {span(fd(n, 'gmm'))} · "
          f"ảnh dựng lại {span([s['recFd'] for s in G[n]['seeds']])}")  # fmt: skip

Với $d = 16$, ảnh dựng lại gần như thật (FD khoảng 2), nhưng mẫu từ $\mathcal N(0, I)$ chỉ đạt khoảng 31, còn mẫu từ
GMM khoảng 8. Phần lớn khoảng cách giữa mẫu và ảnh thật nằm ở chỗ **lấy $z$ ở đâu**, không nằm ở bộ giải mã.

## 9 · Mẹo tái tham số hóa: vì sao không dùng REINFORCE?

Huấn luyện VAE cần gradient theo $\mu$ (và $\sigma$) của $\mathbb E_{z \sim q}[\log p(x \mid z)]$, một kỳ vọng
theo chính phân phối ta đang học. Có hai cách ước lượng nó từ một mẫu $z = \mu + \sigma \odot \varepsilon$:

- **tái tham số hóa:** $\nabla_z \log p(x \mid z)$, lan truyền ngược qua bộ giải mã (hàm `loglik_grad` ở trên);
- **hàm điểm** (score function, REINFORCE): $\log p(x \mid z) \cdot \nabla_\mu \log q(z \mid x) =
  \log p(x \mid z)\cdot \varepsilon / \sigma$. Nó chỉ cần *giá trị* $\log p(x \mid z)$, nên dùng được cả khi không
  lấy đạo hàm theo $z$ được (biến rời rạc, phần thưởng của học tăng cường). Trừ đi một **đường cơ sở**
  $\log p(x \mid z = \mu)$ không làm đổi kỳ vọng, vì $\mathbb E[\varepsilon] = 0$.

Cả ba đều không thiên lệch. Ta đo chúng trên bộ giải mã của VAE 2 chiều, với $\mu$ và $\sigma$ của 20 ảnh demo, mỗi
ảnh 3.000 mẫu $\varepsilon$. Demo đã tính "gradient thật" của từng ảnh từ 200.000 mẫu.

### 🤔 Dự đoán trước
Phương sai của một mẫu REINFORCE (không có đường cơ sở) lớn gấp bao nhiêu lần một mẫu tái tham số hóa: 2, 10, 100
hay hơn thế nữa?

In [ ]:
N_EPS = 3000
est_var, z_worst, err_curve = {"sf": [], "sfb": []}, 0.0, None
for i in range(20):
    x = Xte_bin[SHOW[i]].astype(np.float64)
    eps = np.random.default_rng(300 + i).standard_normal((N_EPS, 2))
    f, g = loglik_grad(dec["vae"], mu20[i] + sd20[i] * eps, x)
    est = {"rp": g, "sf": f[:, None] * eps / sd20[i], "sfb": (f - f_mu[i])[:, None] * eps / sd20[i]}
    var = {k: ((e - e.mean(0)) ** 2).sum(1).mean() for k, e in est.items()}  # vết của ma trận hiệp phương sai
    true = np.array(D20["vae"]["grad"][i]["true"])
    for k, e in est.items():  # trung bình của mỗi ước lượng cách gradient thật bao nhiêu sai số chuẩn?
        z_worst = max(z_worst, np.max(np.abs(e.mean(0) - true) / (e.std(0) / math.sqrt(N_EPS))))
    for k in ("sf", "sfb"):
        est_var[k].append(var[k] / var["rp"])
    if i == 0:  # đường cong sai số theo số mẫu, cho hình bên dưới
        var0, n = var, np.arange(1, N_EPS + 1)
        err_curve = {k: np.linalg.norm(np.cumsum(e, 0) / n[:, None] - true, axis=1) for k, e in est.items()}
ratio = {k: np.array(v) for k, v in est_var.items()}
ref = {k: np.array([q["var"][k] / q["var"]["rp"] for q in D20["vae"]["grad"]]) for k in ("sf", "sfb")}
print(f"cả ba ước lượng khớp gradient thật: lệch lớn nhất {vn(z_worst, 1)} sai số chuẩn")
for k, label in [("sf", "REINFORCE"), ("sfb", "REINFORCE + đường cơ sở")]:
    print(f"{label:>24}: phương sai gấp {vn(ratio[k].min(), 0 if k == 'sf' else 1)}–{vn(ratio[k].max(), 0 if k == 'sf' else 1)} "
          f"lần (trung vị {vn(np.median(ratio[k]), 0 if k == 'sf' else 1)}) · demo, 200.000 mẫu: "
          f"{vn(ref[k].min(), 0 if k == 'sf' else 1)}–{vn(ref[k].max(), 0 if k == 'sf' else 1)} lần")  # fmt: skip
assert z_worst < 5 and np.abs(np.log(ratio["sf"] / ref["sf"])).max() < 0.1
assert np.abs(np.log(ratio["sfb"] / ref["sfb"])).max() < 0.25 and ratio["sf"].min() > 500

Gấp hàng nghìn lần: 768–49.780 lần tùy ảnh, khớp với số mà demo tính từ 200.000 mẫu. Phương sai gấp $r$ lần nghĩa là
phải lấy $r$ mẫu REINFORCE mới chính xác bằng **một** mẫu tái tham số hóa. Hình dưới vẽ, cho ảnh demo đầu tiên (một số
7), khoảng cách từ trung bình của $n$ mẫu đầu tiên đến gradient thật, theo $n$ (hai trục log). Đường nét đứt xám là
độ dốc $1/\sqrt n$ mà mọi ước lượng không thiên lệch đều theo; khác nhau chỉ là độ cao.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
n = np.arange(1, N_EPS + 1)
style = {
    "rp": ("tái tham số hóa", BLUE, "-"),
    "sfb": ("REINFORCE + đường cơ sở", GREEN, (0, (6, 2))),
    "sf": ("REINFORCE", ORANGE, (0, (1, 1.5))),
}
for k, (label, color, ls) in style.items():
    ax.plot(n, err_curve[k], color=color, ls=ls, lw=2)
    ax.text(N_EPS * 1.08, err_curve[k][-1], label.replace(" + ", " +\n"), color=color, va="center", fontsize=9)
ax.plot(n, err_curve["rp"][9] * np.sqrt(10 / n), color=GREY, ls="--", lw=1, label="độ dốc 1/√n")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set(xlim=(1, N_EPS * 15), xlabel="số mẫu ε đã lấy trung bình", ylabel="‖trung bình − gradient thật‖")
ax.set_title(
    f"Ước lượng gradient theo μ cho một ảnh (độ dài gradient thật {vn(np.linalg.norm(D20['vae']['grad'][0]['true']), 1)})"
)
ax.legend(frameon=False, fontsize=8, loc="lower left")
vn_axes(fig)
plt.show()
print(" · ".join(f"{style[k][0]} sau {vn(N_EPS, 0)} mẫu: lệch {vn(err_curve[k][-1], 2)}" for k in style))
print(f"sai số kỳ vọng √(phương sai / n): tái tham số hóa, 1 mẫu: {vn(math.sqrt(var0['rp']), 1)} · REINFORCE, "
      f"{vn(N_EPS, 0)} mẫu: {vn(math.sqrt(var0['sf'] / N_EPS), 1)} · log p(x | z = μ) của ảnh này: {vn(f_mu[0], 1)} nat")  # fmt: skip

Sau 3.000 mẫu, REINFORCE vẫn lệch xa hơn cả độ dài của gradient thật. Tính theo phương sai, trung bình của 3.000 mẫu
REINFORCE (sai số kỳ vọng 34,3) còn kém một mẫu tái tham số hóa duy nhất (22,3). Lý do: REINFORCE chỉ dùng
**một con số** $\log p(x \mid z)$ (−77,8 nat ở
$z = \mu$ cho ảnh này) nhân với $\varepsilon/\sigma$, mà $\sigma$ của ảnh này chỉ là (0,092; 0,048), nên mỗi
mẫu là một vectơ khổng lồ chỉ đúng *trung bình*. Tái tham số hóa dùng đạo hàm $\partial \log p / \partial z$ mà lan
truyền ngược cho gần như miễn phí. Đường cơ sở trừ đi phần lớn con số −80 đó, nên đỡ nhiều, nhưng phương sai vẫn
gấp 3,6–19,1 lần.

Khi huấn luyện thật bằng từng cách (các lần chạy của demo, 20.000 bước, 3 seed), khoảng cách đó thành ra số nat:

In [ ]:
print(
    f"{'':>6} {'tái tham số hóa':>16} {'REINFORCE + đường cơ sở':>25} {'REINFORCE':>14}   (−ELBO trên ảnh kiểm tra, nat)"
)
for d, rp in [(2, "vae2"), (16, "d16"), (50, "b1")]:
    cells = [span([-s["elbo"] for s in G[n]["seeds"]], 1) for n in (rp, f"sfb{d}", f"sf{d}")]
    med = {n: np.median([-s["elbo"] for s in G[n]["seeds"]]) for n in (rp, f"sfb{d}", f"sf{d}")}
    print(f"d = {d:>2} {cells[0]:>16} {cells[1]:>25} {cells[2]:>14}   trung vị kém hơn tái tham số hóa: "
          f"{vn(med[f'sfb{d}'] - med[rp], 1)} và {vn(med[f'sf{d}'] - med[rp], 1)} nat")  # fmt: skip

Có đường cơ sở, REINFORCE vẫn học được một VAE dùng được, chỉ kém 3,2–8,6 nat (trung vị); không có đường cơ sở thì
kém 49,3–66,8 nat, và các seed khác nhau rất xa. Mẹo tái tham số hóa không phải cách *duy nhất* để lấy gradient qua
bước lấy mẫu; nó là
cách có **phương sai thấp**. Với biến tiềm ẩn rời rạc, không viết được $z = \mu + \sigma \odot \varepsilon$, nên người
ta vẫn dùng REINFORCE có đường cơ sở, hoặc các xấp xỉ như Gumbel-softmax hay straight-through (VQ-VAE).

## 10 · ELBO chỉ là một cận dưới của $\log p(x)$

Với β = 1, $-\mathcal L$ là **ELBO**: $\log p(x) \ge \mathbb E_q[\log p(x \mid z)] - D_{\mathrm{KL}}(q \,\|\, p)$.
Khoảng hở chính là $D_{\mathrm{KL}}\big(q(z \mid x) \,\|\, p(z \mid x)\big)$, mức sai của đám mây $q$ so với phân phối
hậu nghiệm thật. Lấy $K$ mẫu $z_k \sim q$ thì có một cận chặt hơn (IWAE, Burda và cộng sự, 2016):

$$\log p(x) \;\ge\; \mathbb E\Big[\log \tfrac1K \textstyle\sum_{k=1}^{K} \tfrac{p(x \mid z_k)\, p(z_k)}{q(z_k \mid x)}\Big],$$

cận này tăng dần theo $K$ và tiến về $\log p(x)$. Ô dưới tính nó trên 20 ảnh demo với bộ giải mã của VAE 2 chiều,
chia 1.000 mẫu thành các nhóm $K$ = 1, 10, 100, 1.000.

In [ ]:
ladder = []
for i in range(20):
    x = Xte_bin[SHOW[i]].astype(np.float64)
    eps = np.random.default_rng(500 + i).standard_normal((1000, 2))
    z = mu20[i] + sd20[i] * eps
    f = loglik_grad(dec["vae"], z, x)[0]
    log_w = (
        f - 0.5 * (z**2).sum(1) + 0.5 * (eps**2).sum(1) + np.log(sd20[i]).sum()
    )  # log p(x|z) + log p(z) − log q(z|x)
    kl = 0.5 * (mu20[i] ** 2 + sd20[i] ** 2 - 1 - np.log(sd20[i] ** 2)).sum()
    row = [f.mean() - kl]  # ELBO, với KL dạng đóng
    for K in (1, 10, 100, 1000):
        g = log_w.reshape(-1, K)
        mx = g.max(1, keepdims=True)
        row.append(float((mx[:, 0] + np.log(np.exp(g - mx).mean(1))).mean()))
    ladder.append(row)
ladder = np.array(ladder).mean(0)
print("20 ảnh demo, trung bình (nat): ELBO " + vn(ladder[0], 2) + " · " + " · ".join(
    f"IWAE K = {vn(K, 0)} {vn(v, 2)}" for K, v in zip((1, 10, 100, 1000), ladder[1:])))  # fmt: skip
assert np.all(np.diff(ladder[1:]) > 0)  # nhóm lồng nhau: cận không giảm khi K tăng (bất đẳng thức Jensen)
print(f"IWAE K = 1.000 cao hơn ELBO {vn(ladder[-1] - ladder[0], 2)} nat")
for n, label in [("vae2", "d = 2"), ("b1", "d = 50")]:
    s = G[n]["seeds"]
    print(f"toàn bộ tập kiểm tra, VAE {label} (demo, 3 seed): −ELBO {span([-r['elbo'] for r in s], 1)} nat, "
          f"−IWAE₁₀₀₀ {span([-r['iwae'] for r in s], 1)} nat")  # fmt: skip

IWAE với $K = 1$ là chính ELBO, chỉ khác ở chỗ KL được ước lượng bằng mẫu thay vì dạng đóng, nên hai số gần như
trùng nhau. Mỗi lần tăng $K$, cận nâng lên; với $K = 1.000$, cao hơn ELBO 2,56 nat trên 20 ảnh này. Trên cả tập kiểm
tra, khoảng hở của VAE 50 chiều lớn hơn: −ELBO 95,5–95,7 nat, còn IWAE-1.000 cho 89,7–90,1. Vì vậy cực đại ELBO không
giống hệt cực đại $\log p(x)$: mô hình còn được thưởng khi làm cho hậu nghiệm thật dễ xấp xỉ bằng một đám mây Gauss.

## 11 · β: mã mang bao nhiêu nat, và khi nào VAE sụp

β-VAE nhân KL với β. $D_{\mathrm{KL}}$ là cái giá, tính bằng nat, mà mã trả để mang thông tin về ảnh (trung bình trên
các ảnh, nó bằng chính lượng thông tin đó cộng khoảng cách từ $q(z)$ đến $\mathcal N(0, I)$, mục 8): β càng lớn,
thông tin càng đắt. Demo đã
huấn luyện VAE 50 chiều với β từ 0,03 đến 30 (3 seed mỗi β). Ô dưới in, cho mỗi β: KL và sai số tái tạo (nat trên ảnh
kiểm tra), số **chiều hoạt động** (chiều $u$ có $\mathrm{Var}_x\, \mu_u > 0{,}01$, định nghĩa của Burda và cộng sự,
2016), SSE trên ảnh xám, và FD của mẫu lấy từ $\mathcal N(0, I)$.

In [ ]:
BETAS = lat["setup"]["betas"]


def fb(b):
    """β như demo: 0,03 · 0,3 · 1 · 30."""
    return vn(b, 2 if b < 0.1 else 1 if b < 1 else 0)


BURDA_ACTIVE = 19  # Burda, Grosse & Salakhutdinov (2016), Bảng 1: VAE một tầng ngẫu nhiên 50 chiều, k = 1


def beta_rows(b):
    return G[f"b{b:g}"]["seeds"]


print(f"{'β':>5} {'KL':>13} {'tái tạo':>13} {'chiều':>7} {'SSE':>13} {'FD N(0, I)':>15} {'FD GMM':>13}")
for b in BETAS:
    s = beta_rows(b)
    print(f"{fb(b):>5} {span([r['kl'] for r in s], 1):>13} "
          f"{span([r['rec'] for r in s], 1):>13} {span([r['active'] for r in s], 0):>7} {span([r['sse'] for r in s], 1):>13} "
          f"{span([r['prior']['fd'] for r in s], 0):>15} {span([r['gmm']['fd'] for r in s], 0):>13}")  # fmt: skip

Hình dưới vẽ cùng bảng trên mặt phẳng **tỉ lệ – méo** (Alemi và cộng sự, 2018): trục ngang là KL (tỉ lệ, *rate*:
cận trên của số nat mã mang về ảnh), trục dọc là sai số tái tạo; mỗi chấm là một seed, ghi β bên cạnh. Các đường xám nét đứt là những điểm có cùng
−ELBO = tái tạo + KL. β trượt dọc theo một đường cong đánh đổi; β = 1 là điểm chạm đường −ELBO thấp nhất.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.4))
for c in (100, 150, 200):
    k = np.linspace(0, c, 2)
    ax.plot(k, c - k, color=GREY, ls="--", lw=0.8)
    ax.text(c * 0.98, 2, f"−ELBO = {c}", color=GREY, fontsize=8, ha="right")
for b in BETAS:
    s = beta_rows(b)
    ax.scatter([r["kl"] for r in s], [r["rec"] for r in s], s=30, color=BLUE, zorder=3)
    ax.annotate(f"β = {fb(b)}", (s[0]["kl"], s[0]["rec"]), xytext=(6, 4),
                textcoords="offset points", fontsize=9)  # fmt: skip
ax.set(xlabel="KL (nat): tỉ lệ", ylabel="−log p(x|z): nat tái tạo (méo)", xlim=(-5, 160), ylim=(0, 215))
ax.set_title("VAE 50 chiều theo β (3 seed): mã càng ít nat, tái tạo càng tệ")
vn_axes(fig)
plt.show()
elbo = {b: [-r["elbo"] for r in beta_rows(b)] for b in BETAS}
print(
    "−ELBO (nat, β = 1 trong công thức, để so được giữa các β): "
    + " · ".join(f"β = {fb(b)}: {span(v, 1)}" for b, v in elbo.items())
)
print(f"Burda và cộng sự (2016), VAE 50 chiều trên MNIST: {BURDA_ACTIVE}/50 chiều hoạt động")

Đọc bảng từ trên xuống:
- **β nhỏ** (0,03–0,1): KL hơn 100 nat, cả 50 chiều đều dùng, ảnh dựng lại gần hoàn hảo. Nhưng mẫu từ
  $\mathcal N(0, I)$ rất tệ (FD trên 100): KL quá rẻ, các đám mây $q(z \mid x)$ không cần chồng lên nhau, và
  $\mathcal N(0, I)$ rơi vào khoảng trống giữa chúng như với autoencoder. GMM vẫn cho FD 19–23.
- **β = 1** (VAE gốc): −ELBO thấp nhất bảng, và chỉ **19–20 trên 50 chiều** hoạt động. Burda và cộng sự (2016) cũng
  đếm được 19/50 trên MNIST. Các chiều còn lại có KL ≈ 0: bộ mã hóa cho chúng đúng $\mathcal N(0, 1)$ với mọi ảnh,
  và bộ giải mã bỏ qua chúng.
- **β lớn**: KL chỉ vài nat, ảnh mờ hẳn (SSE 30–34 ở β = 10). Với β = 30, KL = 0 và không chiều nào hoạt động:
  mọi ảnh có cùng một mã, và bộ giải mã chỉ vẽ một ảnh trung bình. SSE 52,9 đúng bằng sai số của "không nén gì, chỉ
  ảnh trung bình $m$" ở mục 3 (52,89). Đó là **sụp hậu nghiệm** (posterior collapse): $q(z \mid x) = p(z)$ với mọi $x$.

Mẫu từ $\mathcal N(0, I)$ tốt nhất ở β = 1–3 (trong dao động giữa các seed): đủ KL để các đám mây phủ kín gốc, mà
chưa mất quá nhiều thông tin. β > 1 từng được cho là giúp mỗi chiều mã ứng với một yếu tố tạo ảnh riêng (β-VAE,
Higgins và cộng sự, 2017); Locatello và cộng sự (2019) chỉ ra rằng không có thiên kiến quy nạp hay nhãn thì không
bảo đảm được điều đó. Bảng này chỉ đo đánh đổi giữa số nat và sai số.

### β ẩn trong bộ giải mã Gauss
Nhiều bài hướng dẫn huấn luyện VAE trên ảnh xám bằng "MSE + KL". Một bộ giải mã Gauss với phương sai cố định
$\sigma^2$ cho sai số tái tạo $\lVert \hat x - x\rVert^2 / (2\sigma^2)$ (cộng một hằng số). Nhân cả mất mát với
$2\sigma^2$:

$$\frac{\lVert \hat x - x\rVert^2}{2\sigma^2} + D_{\mathrm{KL}} \;\propto\; \lVert \hat x - x\rVert^2 + 2\sigma^2 D_{\mathrm{KL}}$$

Chọn $\sigma^2$ là chọn một β ngầm, $\beta = 2\sigma^2$, so với tổng bình phương sai số (Doersch, 2016). "Tổng bình
phương + KL" là $\sigma^2 = \tfrac12$. "MSE trung bình trên 784 điểm ảnh + KL" là $\lVert \hat x - x\rVert^2/784 +
D_{\mathrm{KL}}$, tức $2\sigma^2 = 784$, $\sigma^2 = 392$: KL nặng gấp 784 lần.

### 🤔 Dự đoán trước
Ta huấn luyện hai VAE 8 chiều trên ảnh xám, chỉ khác $\sigma^2$: ½ và 392. Với $\sigma^2 = 392$, mô hình dùng bao
nhiêu trong 8 chiều?

In [ ]:
gauss = {}
for s2 in (0.5, 392.0):
    p_g, _ = train(8, True, steps=600, batch=128, rec="gauss", s2=s2)
    gauss[s2] = evaluate(p_g, rec="gauss", s2=s2)
    r = gauss[s2]
    print(f"σ² = {vn(s2, 1 if s2 < 1 else 0):>3}: KL {vn(r['kl'], 3)} nat · {r['active']}/8 chiều hoạt động · "
          f"SSE {vn(r['sse'], 1)} (chỉ ảnh trung bình: {vn(((Xte - m_pca) ** 2).sum(1).mean(), 1)})")  # fmt: skip
assert gauss[392.0]["active"] == 0 and gauss[392.0]["kl"] < 0.01 and gauss[0.5]["active"] == 8
print(f"phương sai lớn nhất của ảnh theo một hướng (trị riêng PCA lớn nhất): λ₁ = {vn(lam[0])}, nhỏ hơn 392 "
      f"{vn(392 / lam[0], 0)} lần; {int((lam > 0.5).sum())} hướng có phương sai lớn hơn ½")  # fmt: skip
for n in ("g0.5", "g392"):
    s = G[n]["seeds"]
    print(f"demo, VAE 50 chiều, 20.000 bước, σ² = {n[1:].replace('.', ',')}: KL {span([abs(r['kl']) for r in s], 2)} nat, "
          f"{span([r['active'] for r in s], 0)}/50 chiều, SSE {span([r['sse'] for r in s], 1)}")  # fmt: skip

Không chiều nào. Với $\sigma^2 = 392$, bộ giải mã tự cho rằng mỗi điểm ảnh có nhiễu độ lệch chuẩn khoảng 20, trong khi
điểm ảnh chỉ nằm trong $[0, 1]$: mọi chi tiết của ảnh đều "chỉ là nhiễu", nên trả KL cho chúng là lỗ. Mô hình chọn
KL = 0, như β = 30 ở trên. Lần chạy ngắn này còn chưa kịp vẽ đúng ảnh trung bình (SSE của nó lớn hơn 52,9); các lần
chạy 20.000 bước của demo thì sụp hoàn toàn và đạt đúng SSE của ảnh trung bình. Với VAE tuyến tính có thể tính đúng
lời giải: nó giống PCA xác suất (Tipping & Bishop, 1999; Lucas và cộng sự, 2019), và một hướng chỉ được giữ lại khi
phương sai của dữ liệu theo hướng đó lớn hơn $\sigma^2$. Ở đây phương sai lớn nhất chỉ là 5,12, nhỏ hơn 392 khoảng 77
lần. Với $\sigma^2 = \tfrac12$, 23 hướng có phương sai lớn hơn ½, và VAE dùng cả 8 chiều.

Vậy ảnh VAE mờ vì nhiều lẽ chồng lên nhau, và KL chỉ là một trong số đó:
- nút thắt hẹp (mục 6: autoencoder và VAE 2 chiều mờ như nhau);
- hình ta xem là trung bình $\mathbb E[x \mid z]$, không phải một mẫu (mục 7);
- phân phối của bộ giải mã: $\sigma^2$ lớn, hay β lớn, làm mã mang ít thông tin.

## 12 · Stable Diffusion: autoencoder để nén, khuếch tán để sinh

Stable Diffusion (Rombach và cộng sự, 2022) dùng đúng ý tưởng của bài này, nhưng chia việc khác đi. Bộ nén của nó là
một autoencoder có KL, biến ảnh 512×512×3 thành một lưới tiềm ẩn 64×64×4. Cấu hình công khai `autoencoder_kl_32x32x4`
của CompVis đặt trọng số KL `kl_weight` = 10⁻⁶, bên cạnh mất mát L1, LPIPS và một bộ phân biệt (GAN, Bài 20). KL nhỏ
như thế chỉ giữ cho mã khỏi trôi đi xa; nó không làm mã có hình $\mathcal N(0, I)$. Mã có độ lệch chuẩn khoảng 5,5,
nên Stable Diffusion nhân nó với `scale_factor` = 0,18215 trước khi đưa vào mô hình khuếch tán, và không bao giờ lấy
mẫu $z$ từ $\mathcal N(0, I)$ rồi giải mã: học phân phối của mã là việc của mô hình khuếch tán (Bài 21). Đó chính là
bài học của mục 8: autoencoder tốt + một mô hình mật độ riêng cho mã.

In [ ]:
H, W, C_img = 512, 512, 3
f, c_lat = 8, 4  # nén 8 lần mỗi chiều không gian, 4 kênh tiềm ẩn (SD 1.x; SD3 và FLUX dùng 16 kênh)
n_img, n_lat = H * W * C_img, (H // f) * (W // f) * c_lat
print(
    f"ảnh {H}×{W}×{C_img} = {vn(n_img, 0)} số → mã {H // f}×{W // f}×{c_lat} = {vn(n_lat, 0)} số: ít hơn {vn(n_img / n_lat, 0)} lần"
)
print(f"16 kênh: {vn((H // f) * (W // f) * 16, 0)} số, ít hơn {vn(n_img / ((H // f) * (W // f) * 16), 0)} lần")
print(f"MNIST ở mục 6: 784 → 2 số, ít hơn {vn(784 / 2, 0)} lần · scale_factor 0,18215 = 1 / {vn(1 / 0.18215, 2)} "
      f"(≈ 1 / {vn(1 / 0.18215, 1)})")  # fmt: skip
assert n_img / n_lat == 48

## 13 · Thử thay đổi

Các ô dưới chạy lại cùng các hàm với tham số khác. Bạn có thể sửa giá trị rồi chạy lại.

### Bao nhiêu thành phần cho hỗn hợp Gauss?
Một GMM với $K = 1$ thành phần chính là phân phối Gauss khớp của mục 8. Ô dưới khớp GMM với $K$ = 1, 3, 10, 30
thành phần trên mã của autoencoder, rồi đo như mục 8: tỉ lệ mẫu rơi vào khoảng trống và chữ số hiếm nhất.
Đổi `KS_GMM` để thử thêm.

In [ ]:
KS_GMM = (1, 3, 10, 30)
ref, held = codes["ae"][:4000], codes["ae"][4000:]
r99 = np.quantile(nearest(held, ref)[0], 0.99)
gmm_live = {}
for K in KS_GMM:
    z = gmm_sample(gmm_fit(ref, K=K), 2000, np.random.default_rng(7))
    dist, j = nearest(z, ref)
    share = np.bincount(code_lab[j], minlength=10) / len(z)
    gmm_live[K] = (dist > r99).mean()
    print(
        f"K = {K:>2}: {pct(gmm_live[K], 1):>5} mẫu rơi vào khoảng trống; chữ số hiếm nhất ({share.argmin()}) chiếm {pct(share.min(), 1)}"
    )
assert gmm_live[1] > 2 * gmm_live[10]

Thêm thành phần thì mẫu bám sát các cụm mã hơn. Nhưng "rơi vào khoảng trống" chỉ là một nửa câu chuyện: một GMM có
thật nhiều thành phần, mỗi thành phần ôm sát một mã, sẽ chỉ chép lại các ảnh đã thấy. Bài 20 đo chuyện chép ảnh này
bằng khoảng cách tới ảnh huấn luyện gần nhất.

### β trên VAE 2 chiều
Ô dưới huấn luyện lại VAE 2 chiều của mục 6 với β = 0,1, 1 và 10 (300 bước mỗi lần, ngắn hơn mục 6), rồi in KL, độ
lệch chuẩn của mã và SSE.

In [ ]:
beta_live = {}
for beta in (0.1, 1.0, 10.0):
    r = beta_live[beta] = evaluate(train(2, True, steps=300, batch=128, beta=beta)[0])
    print(f"β = {fb(beta):>3}: KL {vn(r['kl'], 2)} nat · độ lệch chuẩn của mã ({vn(r['mu'][:, 0].std())}; "
          f"{vn(r['mu'][:, 1].std())}) · SSE {vn(r['sse'])}")  # fmt: skip
assert beta_live[0.1]["kl"] > beta_live[1.0]["kl"] > beta_live[10.0]["kl"]
assert beta_live[0.1]["mu"].std() > beta_live[10.0]["mu"].std()
sses = [r["sse"] for r in beta_live.values()]
assert max(sses) / min(sses) < 1.1

KL giảm khi β tăng, và mã co dần về gốc. SSE của lần chạy ngắn này gần như không đổi: ở 2 chiều, nút thắt vẫn là giới
hạn chính, như mục 6. Với 50 chiều (mục 11), β mới đổi hẳn chất lượng tái tạo. Tăng β lên 30 hay 100 rồi chạy lại
để xem một rồi cả hai chiều tắt hẳn.

## 14 · Giới hạn và bước tiếp theo

- **Mẫu vẫn mờ.** Bộ giải mã Bernoulli coi các điểm ảnh độc lập khi biết $z$; trung bình của chúng mờ, còn mẫu thì
  lấm tấm. Bộ giải mã tự hồi quy (PixelVAE) hay phân cấp (NVAE) sửa được phần này, nhưng chậm hơn nhiều.
- **$\mathcal N(0, I)$ không phủ đúng phân phối của mã.** Mục 8 cho thấy một GMM khớp sau khi huấn luyện cho mẫu tốt
  hơn; Stable Diffusion giao phần này cho một mô hình khuếch tán.
- **Thước đo là ELBO, không phải chất lượng ảnh.** Cực đại ELBO thưởng cho việc giải thích mọi điểm ảnh, kể cả nhiễu,
  chứ không thưởng cho việc trông giống thật.

Một hướng khác bỏ hẳn likelihood: huấn luyện bộ sinh ảnh chỉ bằng một mạng thứ hai đánh giá "thật hay giả". Đó là GAN,
bài tiếp theo.

## Tóm tắt
- Nén tuyến tính tốt nhất là PCA: với k = 30, sai số 13,86 mỗi ảnh (13,87 khi khớp trên 60.000 ảnh, như Hinton &
  Salakhutdinov). Autoencoder tuyến tính học đúng không gian con đó (trùng 99,9%) nhưng không học đúng các trục.
- Lớp phi tuyến nén tốt hơn nhiều: autoencoder sâu còn 3,63–3,69 ở k = 30.
- Ở 2 chiều, autoencoder và VAE mờ như nhau (SSE 27,34–27,65 và 27,26–27,71): ảnh mờ do nút thắt, không do KL.
- Mã của autoencoder có thang đo tùy ý; lấy từ $\mathcal N(0, I)$ thì gần như thiếu hẳn số 1. Mã của VAE gom quanh
  gốc với độ lệch chuẩn gần 1. Nhưng autoencoder + GMM cho mẫu ngang VAE + $\mathcal N(0, I)$ (Ghosh và cộng sự).
- Mẹo tái tham số hóa: phương sai một mẫu nhỏ hơn REINFORCE 768–49.780 lần; cả hai đều không thiên lệch.
- ELBO là cận dưới: IWAE chặt hơn vài nat. β = 1 cho −ELBO tốt nhất và chỉ dùng 19–20/50 chiều; β lớn thì sụp.
- "MSE + KL" ($\sigma^2 = 392$, tức β = 784) sụp hoàn toàn: KL = 0, mọi ảnh thành ảnh trung bình.

## Câu hỏi ôn tập
1. Vì sao autoencoder tuyến tính đạt đúng sai số của PCA, nhưng các hàng của $W_d$ không phải các thành phần chính?
2. Một autoencoder và một VAE 2 chiều có SSE như nhau. Vậy KL đã "mua" được gì?
3. Vì sao lấy $z$ từ một phân phối Gauss khớp đúng trung bình và hiệp phương sai của mã autoencoder vẫn cho mẫu tệ?
4. Cả hai ước lượng gradient đều không thiên lệch. Vì sao REINFORCE lại cần nhiều mẫu hơn hàng nghìn lần?
5. Với β = 30, SSE của VAE đúng bằng 52,9. Con số đó là gì, và vì sao?
6. Một bài hướng dẫn dùng `loss = mse(x_hat, x) + kl` với `mse` lấy trung bình trên 784 điểm ảnh. Điều gì xảy ra, và
   sửa thế nào?

<details><summary>Gợi ý đáp án</summary>

1. Sai số chỉ phụ thuộc vào không gian con mà $W_d$ trải ra: quay các hướng bên trong không gian con (rồi quay ngược
   ở $W_e$) không đổi $\hat x$. Baldi & Hornik (1989) chứng minh cực tiểu là không gian con của PCA, không phải các trục.
2. Chỗ đặt mã: mã của VAE gom quanh gốc ở thang của $\mathcal N(0, I)$, nên lấy $z \sim \mathcal N(0, I)$ rồi giải mã
   cho đủ 10 chữ số. Mã của autoencoder thì trải xa, và $\mathcal N(0, I)$ chỉ phủ một góc của nó.
3. Mã nằm thành cụm và dải, xen khoảng trống. Phân phối Gauss phủ cả khoảng trống, nơi bộ giải mã chưa từng được huấn
   luyện (4,5–7,1% mẫu rơi vào đó). Hỗn hợp Gauss bám theo các cụm nên tốt hơn hẳn.
4. REINFORCE chỉ dùng giá trị $\log p(x \mid z)$, một số lớn nhân với $\varepsilon / \sigma$ với $\sigma$ nhỏ; nó bỏ
   qua đạo hàm $\partial \log p / \partial z$ mà lan truyền ngược tính được. Đường cơ sở giảm phương sai, nhưng vẫn kém.
5. Sai số của ảnh trung bình $m$ (52,89 ở mục 3): KL = 0 nên mã không mang thông tin gì, và bộ giải mã tốt nhất là
   vẽ cùng một ảnh trung bình cho mọi ảnh. Đó là sụp hậu nghiệm.
6. Tương đương $\sigma^2 = 392$, tức β = 784 so với tổng bình phương: KL đắt đến mức VAE sụp. Cộng sai số trên các
   điểm ảnh thay vì lấy trung bình (σ² = ½), hoặc dùng bộ giải mã Bernoulli trên ảnh nhị phân, hoặc học $\sigma^2$.
</details>

## Tìm hiểu thêm
- D. Kingma & M. Welling (2013), [Auto-Encoding Variational Bayes](https://arxiv.org/abs/1312.6114), và
  [An Introduction to Variational Autoencoders](https://arxiv.org/abs/1906.02691) (2019).
- D. Rezende, S. Mohamed & D. Wierstra (2014), [Stochastic Backpropagation](https://arxiv.org/abs/1401.4082).
- G. Hinton & R. Salakhutdinov (2006), [Reducing the Dimensionality of Data with Neural Networks](https://www.cs.toronto.edu/~hinton/absps/science.pdf).
- P. Baldi & K. Hornik (1989), [Neural networks and principal component analysis](https://doi.org/10.1016/0893-6080(89)90014-2).
- C. Doersch (2016), [Tutorial on Variational Autoencoders](https://arxiv.org/abs/1606.05908): mục 2.4.3 về σ như một
  tham số chính quy hóa.
- Y. Burda, R. Grosse & R. Salakhutdinov (2016), [Importance Weighted Autoencoders](https://arxiv.org/abs/1509.00519):
  cận IWAE và chiều hoạt động.
- M. Hoffman & M. Johnson (2016), [ELBO surgery](https://approximateinference.org/2016/accepted/HoffmanJohnson2016.pdf):
  KL trung bình = thông tin của mã + khoảng cách của phân phối chung đến tiên nghiệm.
- A. Alemi và cộng sự (2018), [Fixing a Broken ELBO](https://arxiv.org/abs/1711.00464): mặt phẳng tỉ lệ – méo.
- J. Lucas và cộng sự (2019), [Don't Blame the ELBO!](https://arxiv.org/abs/1911.02469): sụp hậu nghiệm ở VAE tuyến tính.
- B. Dai & D. Wipf (2019), [Diagnosing and Enhancing VAE Models](https://arxiv.org/abs/1903.05789).
- P. Ghosh và cộng sự (2020), [From Variational to Deterministic Autoencoders](https://arxiv.org/abs/1903.12436).
- I. Higgins và cộng sự (2017), [β-VAE](https://openreview.net/forum?id=Sy2fzU9gl); F. Locatello và cộng sự (2019),
  [Challenging Common Assumptions in the Unsupervised Learning of Disentangled Representations](https://arxiv.org/abs/1811.12359).
- R. Rombach và cộng sự (2022), [Latent Diffusion](https://arxiv.org/abs/2112.10752), và
  [cấu hình autoencoder KL-f8](https://github.com/CompVis/stable-diffusion/blob/main/configs/autoencoder/autoencoder_kl_32x32x4.yaml).
- [VAE Explainer](https://xnought.github.io/vae-explainer/) (Bertucci & Endert, 2024): một VAE 2 chiều chạy trong trình duyệt.

**Bài tiếp theo:** [Bài 20 · GAN — mạng đối nghịch](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/gan/): bỏ likelihood,
học sinh ảnh bằng một mạng thứ hai bắt giả.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")